In [3]:
import pandas as pd
import numpy as np

from collections import Counter
from scipy.stats import mannwhitneyu

try:
    from statsmodels.stats.multitest import multipletests
except:
    multipletests = None

from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, confusion_matrix

# =========================
# 0. Load data
# =========================

BASE = "../.."

metab = pd.read_csv(f"{BASE}/data/metabolomics/Metabolomics_maaslined_norun.csv")
meta  = pd.read_csv(f"{BASE}/data/metadata/Metadata_061523.csv")

# =========================
# 1. Transpose metabolomics
# =========================

def transpose_omics(df, prefix):
    feature_col = df.columns[0]
    out = df.set_index(feature_col).T
    out.index.name = "sample_id"
    out.columns = [f"{prefix}__{str(c)}" for c in out.columns]
    return out.reset_index()

metab_t = transpose_omics(metab, "metab")

# =========================
# 2. Metadata tp1
# =========================

meta_tp1 = meta[meta["timepoints"] == "tp1"].copy()
meta_tp1 = meta_tp1.rename(columns={"Unnamed: 0": "sample_id"})

meta_tp1["y"] = meta_tp1["study_ptorhc"].map({
    "MECFS": 1,
    "HC": 0,
    "Control": 0
})

meta_tp1 = meta_tp1.dropna(subset=["y"])
meta_tp1["y"] = meta_tp1["y"].astype(int)

meta_small = meta_tp1[["sample_id", "study_ptorhc", "y"]].copy()

# =========================
# 3. Merge
# =========================

df = meta_small.merge(metab_t, on="sample_id", how="inner")

X_metab = df.drop(columns=["sample_id", "study_ptorhc", "y"])
X_metab = X_metab.apply(pd.to_numeric, errors="coerce")
X_metab = X_metab.dropna(axis=1)

y = df["y"]

print("Final metabolomics matrix:", X_metab.shape)
print(y.value_counts())

# =========================
# 4. Helper: train-only stability selection
# =========================

def get_top_stable_features(X_train, y_train, k=15, n_inner=100, C=0.06, seed=42):
    counter = Counter()

    sss = StratifiedShuffleSplit(
        n_splits=n_inner,
        test_size=0.2,
        random_state=seed
    )

    for sub_idx, _ in sss.split(X_train, y_train):
        X_sub = X_train.iloc[sub_idx]
        y_sub = y_train.iloc[sub_idx]

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("lasso", LogisticRegression(
                penalty="l1",
                solver="liblinear",
                C=C,
                max_iter=5000,
                random_state=seed
            ))
        ])

        model.fit(X_sub, y_sub)

        coefs = model.named_steps["lasso"].coef_[0]
        selected = X_sub.columns[coefs != 0]

        for feat in selected:
            counter[feat] += 1

    stable = pd.DataFrame({
        "metabolite": list(counter.keys()),
        "count": list(counter.values())
    }).sort_values("count", ascending=False)

    stable["frequency"] = stable["count"] / n_inner
    stable["clean_name"] = stable["metabolite"].str.replace("metab__", "", regex=False)
    stable["rank"] = np.arange(1, len(stable) + 1)

    return stable.head(k)["metabolite"].tolist(), stable

# =========================
# 5. Outer repeated discovery:
# each split discovers its own top-15 on train only
# then we count which metabolites repeatedly appear in top-15 panels
# =========================

outer_panel_counter = Counter()
outer_aucs = []
outer_accs = []
outer_bal_accs = []

n_outer = 100
n_inner = 100
panel_size = 15

for outer_seed in range(n_outer):
    X_train, X_test, y_train, y_test = train_test_split(
        X_metab,
        y,
        test_size=0.2,
        stratify=y,
        random_state=outer_seed
    )

    panel, stable_this_split = get_top_stable_features(
        X_train,
        y_train,
        k=panel_size,
        n_inner=n_inner,
        C=0.06,
        seed=outer_seed
    )

    for feat in panel:
        outer_panel_counter[feat] += 1

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(max_iter=5000))
    ])

    model.fit(X_train[panel], y_train)

    probs = model.predict_proba(X_test[panel])[:, 1]
    preds = model.predict(X_test[panel])

    outer_aucs.append(roc_auc_score(y_test, probs))
    outer_accs.append(accuracy_score(y_test, preds))
    outer_bal_accs.append(balanced_accuracy_score(y_test, preds))

# =========================
# 6. Global reproducible top-15 panel
# =========================

global_panel_df = pd.DataFrame({
    "metabolite": list(outer_panel_counter.keys()),
    "outer_count": list(outer_panel_counter.values())
}).sort_values("outer_count", ascending=False)

global_panel_df["outer_frequency"] = global_panel_df["outer_count"] / n_outer
global_panel_df["clean_name"] = global_panel_df["metabolite"].str.replace("metab__", "", regex=False)
global_panel_df["rank"] = np.arange(1, len(global_panel_df) + 1)

top15_panel = global_panel_df.head(15)["metabolite"].tolist()

print("\nRepeated nested discovery performance:")
print("Mean AUC:", np.mean(outer_aucs))
print("Median AUC:", np.median(outer_aucs))
print("Std AUC:", np.std(outer_aucs))
print("Min AUC:", np.min(outer_aucs))
print("Max AUC:", np.max(outer_aucs))
print("Mean accuracy:", np.mean(outer_accs))
print("Mean balanced accuracy:", np.mean(outer_bal_accs))

print("\nGlobal top-15 panel by repeated nested discovery:")
display(global_panel_df.head(15)[["rank", "clean_name", "outer_count", "outer_frequency"]])

# =========================
# 7. Evaluate frozen global top-15 panel over 100 repeated splits
# =========================

frozen_aucs = []
frozen_accs = []
frozen_bal_accs = []

for seed in range(100):
    X_train, X_test, y_train, y_test = train_test_split(
        X_metab,
        y,
        test_size=0.2,
        stratify=y,
        random_state=seed
    )

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(max_iter=5000))
    ])

    model.fit(X_train[top15_panel], y_train)

    probs = model.predict_proba(X_test[top15_panel])[:, 1]
    preds = model.predict(X_test[top15_panel])

    frozen_aucs.append(roc_auc_score(y_test, probs))
    frozen_accs.append(accuracy_score(y_test, preds))
    frozen_bal_accs.append(balanced_accuracy_score(y_test, preds))

print("\nFrozen global top-15 repeated-split performance:")
print("Mean AUC:", np.mean(frozen_aucs))
print("Median AUC:", np.median(frozen_aucs))
print("Std AUC:", np.std(frozen_aucs))
print("Min AUC:", np.min(frozen_aucs))
print("Max AUC:", np.max(frozen_aucs))
print("Mean accuracy:", np.mean(frozen_accs))
print("Mean balanced accuracy:", np.mean(frozen_bal_accs))

# =========================
# 8. Final score formula trained on full data
# only after validation
# =========================

final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=5000))
])

final_model.fit(X_metab[top15_panel], y)

coefs = final_model.named_steps["logreg"].coef_[0]
intercept = final_model.named_steps["logreg"].intercept_[0]

score_formula = pd.DataFrame({
    "metabolite": [m.replace("metab__", "") for m in top15_panel],
    "coefficient": coefs
})

print("\nFinal top-15 score formula trained on full data:")
print("Intercept:", intercept)
display(score_formula)

Final metabolomics matrix: (215, 876)
1    136
0     79
Name: y, dtype: int64

Repeated nested discovery performance:
Mean AUC: 0.7503472222222224
Median AUC: 0.75
Std AUC: 0.06236987053194409
Min AUC: 0.5717592592592592
Max AUC: 0.9282407407407407
Mean accuracy: 0.7095348837209303
Mean balanced accuracy: 0.6818749999999999

Global top-15 panel by repeated nested discovery:


,rank,clean_name,outer_count,outer_frequency
9,1,N-acetylglutamate,99,0.99
5,2,3-(4-hydroxyphenyl)lactate,96,0.96
12,3,N-palmitoyl-sphingosine (d18:1/16:0),87,0.87
13,4,"sphingomyelin (d18:1/24:1, d18:2/24:0)*",85,0.85
0,5,N4-acetylcytidine,83,0.83
1,6,1-stearoyl-2-docosahexaenoyl-GPC (18:0/22:6),81,0.81
4,7,1-methylurate,78,0.78
6,8,vanillylmandelate (VMA),60,0.60
10,9,N-palmitoyl-sphinganine (d18:0/16:0),59,0.59
16,10,"sphingomyelin (d18:2/21:0, d16:2/23:0)*",41,0.41



Frozen global top-15 repeated-split performance:
Mean AUC: 0.8619675925925925
Median AUC: 0.8634259259259259
Std AUC: 0.053265548769346104
Min AUC: 0.7222222222222222
Max AUC: 0.974537037037037
Mean accuracy: 0.8093023255813953
Mean balanced accuracy: 0.7851273148148148

Final top-15 score formula trained on full data:
Intercept: 1.0220574750767504


,metabolite,coefficient
0,N-acetylglutamate,0.910012
1,3-(4-hydroxyphenyl)lactate,-0.771231
2,N-palmitoyl-sphingosine (d18:1/16:0),0.385393
3,"sphingomyelin (d18:1/24:1, d18:2/24:0)*",0.412127
4,N4-acetylcytidine,0.914695
5,1-stearoyl-2-docosahexaenoyl-GPC (18:0/22:6),0.571401
6,1-methylurate,-0.518159
7,vanillylmandelate (VMA),-0.758545
8,N-palmitoyl-sphinganine (d18:0/16:0),0.126074
9,"sphingomyelin (d18:2/21:0, d16:2/23:0)*",-0.137137


In [4]:
target = "metab__1-stearoyl-2-docosahexaenoyl-GPC (18:0/22:6)"

print(
    stability_df.loc[
        stability_df["metabolite"] == target,
        ["rank", "count", "frequency"]
    ]
)

    rank  count  frequency
29    19      6       0.06


In [5]:
corrs = (
    X_metab.corr()[target]
    .abs()
    .sort_values(ascending=False)
)

display(corrs.head(20))

metab__1-stearoyl-2-docosahexaenoyl-GPC (18:0/22:6)            1.000000
metab__1-palmitoyl-2-docosahexaenoyl-GPC (16:0/22:6)           0.846448
metab__1-oleoyl-2-docosahexaenoyl-GPC (18:1/22:6)*             0.650277
metab__3-carboxy-4-methyl-5-propyl-2-furanpropanoate (CMPF)    0.646514
metab__hydroxy-CMPF*                                           0.564905
metab__docosahexaenoate (DHA; 22:6n3)                          0.517924
metab__palmitoyl-docosahexaenoyl-glycerol (16:0/22:6) [1]*     0.479257
metab__1-stearoyl-2-docosahexaenoyl-GPE (18:0/22:6)*           0.445610
metab__eicosapentaenoate (EPA; 20:5n3)                         0.444723
metab__X-13866                                                 0.397898
metab__sphingomyelin (d18:1/24:1, d18:2/24:0)*                 0.369463
metab__1-palmitoyl-2-stearoyl-GPC (16:0/18:0)                  0.357544
metab__1-palmitoyl-2-docosahexaenoyl-GPE (16:0/22:6)*          0.320357
metab__1,5-anhydroglucitol (1,5-AG)                            0

In [6]:
from collections import Counter
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score

# use metabolomics only
X_metab = X_all[[c for c in X_all.columns if c.startswith("metab__")]]

def stability_select(X_train, y_train, k, n_inner=100, C=0.06, seed=42):
    counter = Counter()

    sss = StratifiedShuffleSplit(
        n_splits=n_inner,
        test_size=0.2,
        random_state=seed
    )

    for idx, _ in sss.split(X_train, y_train):
        X_sub = X_train.iloc[idx]
        y_sub = y_train.iloc[idx]

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("lasso", LogisticRegression(
                penalty="l1",
                solver="liblinear",
                C=C,
                max_iter=5000,
                random_state=seed
            ))
        ])

        model.fit(X_sub, y_sub)

        coefs = model.named_steps["lasso"].coef_[0]
        selected = X_sub.columns[coefs != 0]

        for feat in selected:
            counter[feat] += 1

    ranked = pd.DataFrame({
        "feature": list(counter.keys()),
        "count": list(counter.values())
    }).sort_values("count", ascending=False)

    return ranked.head(k)["feature"].tolist(), ranked


all_rows = []
panel_counts_by_k = {k: Counter() for k in range(4, 31)}

for k in range(4, 31):
    aucs = []
    accs = []
    bal_accs = []

    for seed in range(50):   # change to 100 later if you want
        X_train, X_test, y_train, y_test = train_test_split(
            X_metab,
            y,
            test_size=0.2,
            stratify=y,
            random_state=seed
        )

        panel, ranked = stability_select(
            X_train,
            y_train,
            k=k,
            n_inner=100,
            C=0.06,
            seed=seed
        )

        for feat in panel:
            panel_counts_by_k[k][feat] += 1

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("logreg", LogisticRegression(max_iter=5000))
        ])

        model.fit(X_train[panel], y_train)

        probs = model.predict_proba(X_test[panel])[:, 1]
        preds = model.predict(X_test[panel])

        aucs.append(roc_auc_score(y_test, probs))
        accs.append(accuracy_score(y_test, preds))
        bal_accs.append(balanced_accuracy_score(y_test, preds))

    all_rows.append({
        "k": k,
        "mean_auc": np.mean(aucs),
        "median_auc": np.median(aucs),
        "std_auc": np.std(aucs),
        "min_auc": np.min(aucs),
        "max_auc": np.max(aucs),
        "mean_accuracy": np.mean(accs),
        "mean_balanced_accuracy": np.mean(bal_accs)
    })

size_results = pd.DataFrame(all_rows).sort_values("mean_auc", ascending=False)

display(size_results)

NameError: name 'X_all' is not defined